# Compare of each simulation 

In [1]:
# 設定: ドメイン名とID、解析期間を指定
domain_name = 'Bangkok'
domain_id = 'd03'  # 例: 'd01', 'd02' など
varname = 'T2'  # プロットしたい変数名
variables = ['Times', varname]
period_start = '2025-01-01 00:00:00'  # 解析開始時刻
period_end = '2025-01-02 23:00:00'    # 解析終了時刻
target_time = '2025-01-02 12:00:00'  # プロットする時刻（例: 'YYYY-MM-DD HH:MM:SS'）
setting = 'test'


In [4]:
import numpy as np
import xarray as xr
from datetime import datetime
from pathlib import Path

# Convert analysis period to numpy datetime64
period_start64 = np.datetime64(period_start)
period_end64 = np.datetime64(period_end)

# Define data directory and retrieve file list
data_dir = Path(f"../../../data/simulation/{domain_name}/{setting}")
files = sorted(data_dir.glob(f"wrfout_{domain_id}_*"))
print(f"Total files: {len(files)}")

# Filter files based on the analysis period
def extract_start_time(path):
    try:
        timestamp = path.name.split(f"wrfout_{domain_id}_", 1)[1]
        return np.datetime64(datetime.strptime(timestamp, "%Y-%m-%d_%H:%M:%S"))
    except (IndexError, ValueError):
        return None

selected_files = [
    str(p) for p in files if (start_time := extract_start_time(p)) and period_start64 <= start_time <= period_end64
]
file_list = selected_files or [str(p) for p in files]
print(f"Using files: {file_list})")

# Optimize chunking for memory efficiency
chunks = {"Time": 1}
try:
    with xr.open_dataset(file_list[0], engine="netcdf4") as sample:
        chunks.update({dim: 300 for dim in sample.dims if dim in ["south_north", "west_east"]})
except Exception as e:
    print(f"Chunk detection failed: {e}")

# Load data with optimized preprocessing
def preprocess_dataset(ds):
    return ds[[v for v in variables if v in ds] or [varname]]

da = xr.open_mfdataset(
    file_list,
    combine="nested",
    engine="netcdf4",
    decode_times=True,
    parallel=True,
    chunks=chunks,
    preprocess=preprocess_dataset,
)[varname].sel(Time=slice(period_start, period_end))

da


: 

In [ ]:
# 2次元変数（例: 地表面気温 T2）をプロット
import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar

# 指定した時刻のインデックスを取得（厳密一致→なければ最近傍）
times = da['Time'].values
t64 = np.datetime64(target_time)
try:
    time_index = int(np.where(times == t64)[0][0])
except IndexError:
    diffs = np.abs(times - t64)
    time_index = int(np.argmin(diffs))
    print(f'Exact time not found. Using closest: {str(times[time_index])}')

plt.figure(figsize=(8,6))
with ProgressBar():
    da.isel(Time=time_index).plot(cbar_kwargs={'label': da.attrs.get('units', '')})
plt.title(f'{varname} at {target_time}')
plt.show()
